# Documentation de la classe `Phenology`

La classe `Phenology` (`kadi.weather.phenology.Phenology`) se charge d'extraire les moments clés de la saison agricole béninoise : 
- Le **début de saison** (Onset)
- La **fin de saison** (Cessation)
- Les **Degrés-Jours de Croissance** (Growing Degree Days - GDD) pour suivre le développement végétatif.

## 1. Initialisation
Pour instancier l'analyseur phénologique, vous devez fournir :
- Une instance `Location`
- Une série de précipitations (`pandas.Series`)
- Un `DataFrame` de températures (`temperature_min`, `temperature_max`)

In [1]:
# Changer de répertoire
import os
from pathlib import Path
print(Path.cwd())
new_dir = Path("~/Bureau/kadipy/").expanduser().resolve()
os.chdir(new_dir)
print(Path.cwd())

from kadi.weather.location import Location
from kadi.weather.data import WeatherData
from kadi.weather.phenology import Phenology
import pandas as pd

# Récupération de l'historique météo pour initialiser la classe
loc = Location(latitude=9.3333, longitude=2.6333, name='Parakou') # Zone Nord
weather = WeatherData(loc)
hist = weather.fetch_historical(months_back=12)

pheno = Phenology(loc, hist['precipitation'], hist[['temperature_min', 'temperature_max']])
print("Analyseur phénologique initialisé.")

/home/dels/Bureau/kadipy/docs/weather
/home/dels/Bureau/kadipy
Analyseur phénologique initialisé.


## 2. Détection du Début de Saison (`onset`)
**Méthode :** `onset(threshold_days_after: int = 120) -> dict`

Le grand défi de l'analyse agro-climatique est que la définition du "début" des pluies dépend fortement de la latitude :
- Pour la **Zone Nord** (régime unimodal, comme Parakou ou Natitingou), KadiPy utilise l'algorithme de **Sivakumar**. Cet algorithme cherche 20 mm de pluie tombée sur 3 jours consécutifs, à condition qu'il n'y ait pas de période sèche (plus de 7 jours consécutifs sans pluie) dans les 30 jours suivants (ce qui tuerait les jeunes pousses).
- Pour les **Zones Sud et Centre** (régime bimodal, comme Cotonou), KadiPy utilise un algorithme **Hybride Walter-Anyadike**, car les critères du Nord ne fonctionnent pas sur les pluies de transition.

In [2]:
onset_info = pheno.onset()
print("Algorithme utilisé :", onset_info['algorithm'])
print("Date de démarrage estimée :", onset_info['onset_date'])

Algorithme utilisé : Sivakumar
Date de démarrage estimée : 2026-05-03


## 3. Détection de la Fin de Saison (`cessation`)
**Méthode :** `cessation() -> dict`

La fin de saison est calculée de manière simplifiée dans ce module : le système balaie les données à l'envers depuis la fin de l'année (après septembre) et détecte le dernier jour avant que le cumul des précipitations restantes de l'année ne descende sous 20 mm.

In [3]:
cessation_info = pheno.cessation()
print("Date de fin de saison :", cessation_info['cessation_date'])
print("Durée de la saison (jours) :", cessation_info['duration_days'])

Date de fin de saison : 2025-10-14
Durée de la saison (jours) : 99


In [4]:
cessation_info

{'cessation_date': '2025-10-14',
 'duration_days': 99,
 'total_rainfall': 581.5000000000001,
 'zone': 'Nord'}

## 4. Degrés-Jours de Croissance (`growing_degree_days`)
**Méthode :** `growing_degree_days(crop: str, start_date: str, end_date: str) -> dict`

Au lieu de compter uniquement le temps, les plantes se développent en fonction de l'accumulation de la chaleur (GDD). Le calcul se base sur l'équation : `GDD = T_moyenne - T_base` (avec une température de base par défaut de 10°C pour le maïs, ou 14°C pour le manioc).

L'accumulation permet d'estimer le stade de développement végétatif (*vegetative*, *flowering*, *maturity*).

In [5]:
# Calcul pour le maïs semé il y a 60 jours
end_date = pd.Timestamp.now()
start_date = end_date - pd.Timedelta(days=60)

gdd = pheno.growing_degree_days(
    crop='maize', 
    start_date=start_date.strftime('%Y-%m-%d'), 
    end_date=end_date.strftime('%Y-%m-%d')
)

print(f"Culture : {gdd['crop']}")
print(f"GDD accumulés : {gdd['gdd_accumulated']}")
print(f"Pourcentage d'avancement du cycle : {gdd['pct_cycle']}%")
print(f"Stade phénologique estimé : {gdd['phenology_stage']}")

Culture : maize
GDD accumulés : 1057.6
Pourcentage d'avancement du cycle : 81%
Stade phénologique estimé : tasseling/flowering
